# Week 11 — Community Detection with a Graph Neural Network

**Theme:** Graph neural networks (GNN)

Not all data is a table or a grid of pixels — a social network, a molecule, or
a road map is a **graph**: nodes connected by edges. A Graph Neural Network
learns by having each node repeatedly "gather information from its neighbors."

**Dataset:** Zachary's Karate Club — a famous small social network. In the
1970s, a real karate club split into two factions after a conflict between the
instructor ("Mr. Hi") and the club president ("Officer"). We'll try to predict
which faction each of the 34 members ended up in, using **only the friendship
graph** and a *handful* of known labels.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

In [ ]:
G = nx.karate_club_graph()
n_nodes = G.number_of_nodes()

# Ground-truth faction each member ended up joining (0 = Mr. Hi, 1 = Officer)
true_labels = np.array([0 if G.nodes[i]["club"] == "Mr. Hi" else 1 for i in G.nodes])

pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(6, 5))
nx.draw(G, pos, node_color=true_labels, cmap="coolwarm", with_labels=True,
        node_size=400, font_size=8, font_color="white")
plt.title("Karate Club Graph (color = which faction they actually joined)")
plt.show()

## 1. Build the (normalized) adjacency matrix

A GCN layer updates each node's representation by averaging its neighbors'
representations. To do that with one matrix multiplication, we build a
**normalized adjacency matrix** `A_norm`, where `A_hat = A + I` (every node
counts itself as its own neighbor too) and we scale by node degree so
high-degree "hub" nodes don't dominate.

In [ ]:
A = nx.to_numpy_array(G)                       # plain adjacency matrix (34x34)
A_hat = A + np.eye(n_nodes)                     # add self-loops
degree = A_hat.sum(axis=1)                      # how many neighbors (+self) each node has
D_inv_sqrt = np.diag(1.0 / np.sqrt(degree))
A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt        # symmetric normalization

A_norm = torch.tensor(A_norm, dtype=torch.float32)
print("A_norm shape:", A_norm.shape)

## 2. A GCN layer, in one line

A single GCN layer is: `A_norm @ (X @ W)` — mix each node's features with its
neighbors' features (`A_norm @ ...`), after applying a learnable linear
transform (`X @ W`). Stack a few of these, with a non-linearity in between,
and nodes gradually "absorb" information from further and further away in the graph.

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, A_norm, X):
        return A_norm @ self.linear(X)


class GCN(nn.Module):
    def __init__(self, n_nodes, hidden_dim=4, embed_dim=2, n_classes=2):
        super().__init__()
        self.gcn1 = GCNLayer(n_nodes, hidden_dim)
        self.gcn2 = GCNLayer(hidden_dim, embed_dim)   # embed_dim=2 so we can plot it directly
        self.classifier = nn.Linear(embed_dim, n_classes)

    def forward(self, A_norm, X):
        h = F.relu(self.gcn1(A_norm, X))
        embedding = self.gcn2(A_norm, h)   # 2D embedding, used both for classifying AND plotting
        logits = self.classifier(embedding)
        return logits, embedding

# No real "features" per node -- we use an identity matrix, so the model has
# to learn everything purely from graph structure, not pre-given attributes.
X = torch.eye(n_nodes)

model = GCN(n_nodes)
print(model)

## 3. What does an *untrained* GCN already see?

Even before any training, a GCN's structure alone tends to place connected
nodes near each other in its embedding space, because averaging with
neighbors is baked into the architecture.

In [ ]:
model.eval()
with torch.no_grad():
    _, embedding = model(A_norm, X)
embedding = embedding.numpy()

plt.figure(figsize=(5, 5))
plt.scatter(embedding[:, 0], embedding[:, 1], c=true_labels, cmap="coolwarm", s=100, edgecolor="k")
for i in range(n_nodes):
    plt.annotate(str(i), (embedding[i, 0], embedding[i, 1]), fontsize=7)
plt.title("2D Embedding BEFORE Training (untrained GCN)")
plt.show()

## 4. Train with only 4 labeled nodes

This is **semi-supervised** learning: we only reveal the true faction for a
handful of nodes (2 from each side) and let the loss push the model to be
consistent with the graph structure for everyone else.

In [ ]:
labeled_nodes = torch.tensor([0, 1, 33, 32])  # 0,1: Mr. Hi's close circle; 33,32: Officer's close circle
labeled_targets = torch.tensor(true_labels[labeled_nodes.numpy()], dtype=torch.long)
print("Labeled nodes:", labeled_nodes.tolist())
print("Their true factions:", labeled_targets.tolist())

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

losses = []
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    logits, _ = model(A_norm, X)
    loss = criterion(logits[labeled_nodes], labeled_targets)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 40 == 0:
        print(f"epoch {epoch+1}: loss={loss.item():.4f}")

plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("Training Loss (on the 4 labeled nodes)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

## 5. Evaluate: did it recover the factions for the *unlabeled* members?

In [ ]:
model.eval()
with torch.no_grad():
    logits, embedding_trained = model(A_norm, X)
predicted = logits.argmax(dim=1).numpy()
embedding_trained = embedding_trained.numpy()

unlabeled_mask = np.ones(n_nodes, dtype=bool)
unlabeled_mask[labeled_nodes.numpy()] = False
accuracy = (predicted[unlabeled_mask] == true_labels[unlabeled_mask]).mean()
print(f"Accuracy on the 30 UNLABELED members: {accuracy:.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(embedding_trained[:, 0], embedding_trained[:, 1],
                c=true_labels, cmap="coolwarm", s=100, edgecolor="k")
axes[0].scatter(embedding_trained[labeled_nodes, 0], embedding_trained[labeled_nodes, 1],
                facecolors="none", edgecolors="black", s=250, linewidths=2, label="labeled nodes")
axes[0].set_title("2D Embedding AFTER Training")
axes[0].legend()

correct = predicted == true_labels
node_colors = ["seagreen" if c else "crimson" for c in correct]
nx.draw(G, pos, ax=axes[1], node_color=node_colors, with_labels=True,
        node_size=400, font_size=8, font_color="white")
axes[1].set_title("Green = correctly predicted faction, Red = wrong")
plt.tight_layout()
plt.show()

## Try it yourself

1. **Fewer labels.** Reduce `labeled_nodes` to just `[0, 33]` (one per class)
   — does accuracy hold up? This is the same experiment from the original GCN
   paper's famous demo.
2. **Deeper network.** Add a third `GCNLayer` — does accuracy improve, get
   worse, or stay the same? (Very deep GCNs tend to "over-smooth" — all nodes'
   embeddings start to look the same.)
3. **Compare to Week 4/5.** The 2D embedding plot looks a lot like the PCA and
   K-Means plots from Weeks 4-5 — what's different about *how* this 2D
   representation was learned (hint: think about what information PCA uses
   vs. what the GCN uses)?
4. **No self-loops.** Remove `+ np.eye(n_nodes)` from `A_hat` and re-run —
   why does training get much less stable without self-loops?

---
## 🎯 캡스톤: 신입생 친구 추천 -- 캠퍼스 소셜 그래프 GNN

가상의 캠퍼스 친구 관계 그래프(학생 45명, 3개의 잠재적 친목 그룹)를 드립니다. 몇 명만 그룹이 라벨링되어 있고 나머지는 모릅니다. 위에서 만든 것과 **똑같은 방식**(GCN 레이어를 직접 쌓기)으로 나머지 학생들의 그룹을 예측하고, **여러분 자신을 새로운 노드로 그래프에 추가**해서 어떤 그룹에 배정되는지 확인해보세요.

**확장 아이디어:** 실제 동아리/스터디 모임의 친구 관계(누가 누구와 자주 어울리는지)와 소속 그룹을 기록해서 그래프를 만들면, 이 코드 그대로 "신입생 그룹 추천"에 쓸 수 있습니다.

In [ ]:
# 더미 캠퍼스 소셜 그래프 생성 (실행만 하면 됩니다)
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

rng = np.random.default_rng(23)
n_per_group = 15
group_names = ["운동/스포츠 동아리", "게임/IT 동아리", "음악/공연 동아리"]
n_nodes = n_per_group * len(group_names)

campus_true_group = np.repeat(np.arange(len(group_names)), n_per_group)

# 같은 그룹끼리는 친구가 될 확률이 높고, 다른 그룹끼리는 낮음 (stochastic block model과 유사)
p_within, p_between = 0.35, 0.03
campus_G = nx.Graph()
campus_G.add_nodes_from(range(n_nodes))
for i in range(n_nodes):
    for j in range(i + 1, n_nodes):
        p = p_within if campus_true_group[i] == campus_true_group[j] else p_between
        if rng.random() < p:
            campus_G.add_edge(i, j)

pos = nx.spring_layout(campus_G, seed=1)
plt.figure(figsize=(6, 5))
nx.draw(campus_G, pos, node_color=campus_true_group, cmap="Set1", node_size=150, with_labels=False)
plt.title("Campus Friendship Graph (color = true group, for reference only)")
plt.show()

# 실제로는 이렇게 그룹을 다 알 수 없습니다 -- 몇 명만 라벨을 공개합니다.
campus_labeled_nodes = [0, 1, n_per_group, n_per_group + 1, 2 * n_per_group, 2 * n_per_group + 1]
print("공개된 라벨:", {n: group_names[campus_true_group[n]] for n in campus_labeled_nodes})

### 여러분의 과제

1. Week 11 본문처럼 `campus_G`의 정규화된 인접행렬 `A_norm`을 만드세요. (self-loop 추가 + degree로 정규화)
2. 위에서 정의한 `GCN` 클래스를 그대로 재사용해서(또는 새로 정의해서) `campus_labeled_nodes`(6명)만 라벨로 사용해 학습시키세요. (`n_classes=3`으로 바꿔야 합니다.)
3. 라벨이 없던 나머지 39명에 대한 예측 정확도를 계산하세요 (진짜 서비스라면 알 수 없지만, 여기서는 `campus_true_group`으로 확인할 수 있습니다).
4. **가장 중요한 부분:** 그래프에 새로운 노드(번호 `n_nodes`, 즉 "나")를 하나 추가하고, 좋아하는 그룹의 학생 2~3명과 친구 관계(엣지)를 연결한 뒤, 그래프 전체를 다시 GCN에 넣어 "내가 어느 동아리에 잘 맞을지" 예측해보세요. **주의:** 노드를 추가하면 `A_norm`도 새로 계산해야 하고, 입력 특성(단위행렬)의 크기도 `n_nodes+1`로 바뀌어야 합니다.

In [ ]:
# TODO 1: campus_G로부터 A_norm(정규화된 인접행렬)을 만드세요.


# TODO 2: GCN(n_classes=3)을 정의하고 campus_labeled_nodes만 사용해 학습시키세요.


# TODO 3: 라벨이 없던 나머지 39명에 대한 예측 정확도를 계산하세요.


# TODO 4: "나"를 새 노드로 추가하고 친구 관계를 연결한 뒤, 그래프를 다시 만들어 내가 어느 그룹에 배정되는지 예측해보세요.
my_friend_edges = []  # 예: [(0, n_nodes), (3, n_nodes)]  -- (기존 학생 번호, 나의 새 노드 번호)